# Aula 6 — Transformers: Self-Attention e Encoder (Keras/TensorFlow)

**CCM-109 — Deep Learning · Prof. Ronaldo Prati (UFABC)**

Este notebook implementa o Transformer passo a passo com Keras:

1. **Self-attention do zero** — produto interno, softmax, média ponderada
2. **Positional encoding** — embeddings sinusoidais
3. **Multi-head attention** — camada Keras customizada
4. **Bloco Encoder completo** — MHA + Add&Norm + FFN
5. **Aplicação: NER** — classificação token a token com CoNLL-2003
6. **Visualização de atenção**
7. **Comparação com BiLSTM**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/REPO/blob/main/notebooks/lec06-transformers-keras.ipynb)

## 0. Setup

In [ ]:
# !pip install datasets seaborn --quiet


In [ ]:
import tensorflow as tf
from tensorflow import keras
import numpy as np
import math
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

print(f'TensorFlow: {tf.__version__}')
print(f'GPU disponível: {len(tf.config.list_physical_devices("GPU")) > 0}')
tf.random.set_seed(42)
np.random.seed(42)

## 1. Self-Attention do Zero

Implementação manual para entender cada passo.

$$\text{Attention}(W) = \text{softmax}\!\left(\frac{WW^\top}{\sqrt{d}}\right)W$$

In [ ]:
def self_attention_manual(W):
    """
    Self-attention simplificada (sem projeções Q/K/V).
    W: tensor (N, d)
    Retorna: saída (N, d), pesos de atenção (N, N)
    """
    d = tf.cast(tf.shape(W)[-1], tf.float32)
    scores = tf.matmul(W, W, transpose_b=True) / tf.sqrt(d)  # (N, N)
    attn   = tf.nn.softmax(scores, axis=-1)                   # (N, N)
    out    = tf.matmul(attn, W)                               # (N, d)
    return out, attn


# Exemplo da aula: train / station₁ / radio / station₂
words = ['train', 'sta₁', 'radio', 'sta₂']
W = tf.constant([
    [0.9, 0.1],
    [0.6, 0.6],
    [0.1, 0.9],
    [0.6, 0.6],   # idêntico a sta₁!
], dtype=tf.float32)

out, attn = self_attention_manual(W)

print('Embeddings originais:')
for w, e in zip(words, W.numpy()):
    print(f'  {w:6s}: {e.tolist()}')

print('\nMatriz de atenção:')
print(f"{'':8s}" + ''.join(f'{w:7s}' for w in words))
for i, w in enumerate(words):
    print(f'  {w:6s}: ' + '  '.join(f'{attn[i,j]:.3f}' for j in range(len(words))))

print('\nRepresentações contextuais:')
for w, e in zip(words, out.numpy()):
    print(f'  {w:6s}: {[round(x,3) for x in e.tolist()]}')

print('\n⚠️  sta₁ e sta₂ produzem representações idênticas — motivação para Positional Encoding!')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.heatmap(attn.numpy(), annot=True, fmt='.3f', cmap='Blues',
            xticklabels=words, yticklabels=words, ax=axes[0])
axes[0].set_title('Pesos de atenção\n(sem projeções Q/K/V)')
axes[0].set_xlabel('atende a →'); axes[0].set_ylabel('palavra alvo')

colors = ['#3b82f6', '#eab308', '#a855f7', '#eab308']
for i, (w, c) in enumerate(zip(words, colors)):
    axes[1].annotate('', xy=out.numpy()[i], xytext=[0,0],
                     arrowprops=dict(arrowstyle='->', color=c, lw=2))
    axes[1].annotate('', xy=W.numpy()[i], xytext=[0,0],
                     arrowprops=dict(arrowstyle='->', color=c, lw=1, linestyle='--', alpha=0.4))
    axes[1].text(out.numpy()[i,0]+0.01, out.numpy()[i,1]+0.01, f"{w}'")
    axes[1].text(W.numpy()[i,0]+0.01, W.numpy()[i,1]+0.01, w, alpha=0.4)

axes[1].set(xlim=(-0.1,1.1), ylim=(-0.1,1.1),
            title='Embeddings: originais (tracejado) vs. contextuais (sólido)',
            xlabel='d₁', ylabel='d₂')
axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 2. Positional Encoding Sinusoidal

$$PE(pos, 2k) = \sin\!\left(\frac{pos}{10000^{2k/d}}\right), \quad PE(pos, 2k+1) = \cos\!\left(\frac{pos}{10000^{2k/d}}\right)$$

In [ ]:
class PositionalEncoding(keras.layers.Layer):
    def __init__(self, d_model, max_len=512, **kwargs):
        super().__init__(**kwargs)
        # Pré-computar tabela (max_len, d_model)
        positions = np.arange(max_len)[:, np.newaxis]          # (max_len, 1)
        dims      = np.arange(0, d_model, 2)[np.newaxis, :]    # (1, d_model/2)
        div       = np.exp(dims * (-math.log(10000.0) / d_model))
        pe = np.zeros((max_len, d_model))
        pe[:, 0::2] = np.sin(positions * div)
        pe[:, 1::2] = np.cos(positions * div)
        self.pe = tf.constant(pe[np.newaxis], dtype=tf.float32)  # (1, max_len, d)

    def call(self, x):
        # x: (batch, seq_len, d_model)
        return x + self.pe[:, :tf.shape(x)[1]]


# Visualização
d_model, max_len = 64, 100
pe_layer = PositionalEncoding(d_model, max_len)
pe_matrix = pe_layer.pe[0].numpy()  # (max_len, d_model)

plt.figure(figsize=(14, 4))
plt.subplot(1, 2, 1)
plt.imshow(pe_matrix.T, aspect='auto', cmap='RdBu', origin='lower', extent=[0, max_len, 0, d_model])
plt.colorbar(); plt.xlabel('Posição'); plt.ylabel('Dimensão'); plt.title(f'Positional Encoding sinusoidal (d={d_model})')

plt.subplot(1, 2, 2)
for dim in [0, 1, 4, 5, 20, 21]:
    plt.plot(pe_matrix[:50, dim], label=f'd={dim}')
plt.xlabel('Posição'); plt.ylabel('Valor'); plt.title('50 posições — 6 dimensões')
plt.legend(fontsize=8, ncol=2); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 3. Multi-Head Attention com Projeções Q, K, V

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

Keras já oferece `keras.layers.MultiHeadAttention`. Vamos primeiro implementar do zero, depois usar a versão nativa.

In [ ]:
class MultiHeadAttention(keras.layers.Layer):
    """Multi-Head Attention implementado do zero em Keras."""
    def __init__(self, d_model, num_heads, dropout=0.1, **kwargs):
        super().__init__(**kwargs)
        assert d_model % num_heads == 0
        self.num_heads = num_heads
        self.d_model   = d_model
        self.d_k       = d_model // num_heads

        self.W_q = keras.layers.Dense(d_model, use_bias=False)
        self.W_k = keras.layers.Dense(d_model, use_bias=False)
        self.W_v = keras.layers.Dense(d_model, use_bias=False)
        self.W_o = keras.layers.Dense(d_model, use_bias=False)
        self.drop = keras.layers.Dropout(dropout)
        self.last_attn = None  # para visualização

    def split_heads(self, x, B):
        # (B, N, d_model) → (B, h, N, dk)
        x = tf.reshape(x, (B, -1, self.num_heads, self.d_k))
        return tf.transpose(x, perm=[0, 2, 1, 3])

    def call(self, x, mask=None, training=False):
        B = tf.shape(x)[0]
        Q = self.split_heads(self.W_q(x), B)  # (B, h, N, dk)
        K = self.split_heads(self.W_k(x), B)
        V = self.split_heads(self.W_v(x), B)

        # Scaled dot-product
        scores = tf.matmul(Q, K, transpose_b=True) / math.sqrt(self.d_k)  # (B,h,N,N)
        if mask is not None:
            scores += mask * -1e9
        attn = tf.nn.softmax(scores, axis=-1)
        self.last_attn = attn
        attn = self.drop(attn, training=training)

        out = tf.matmul(attn, V)                                # (B, h, N, dk)
        out = tf.transpose(out, perm=[0, 2, 1, 3])             # (B, N, h, dk)
        out = tf.reshape(out, (B, -1, self.d_model))           # (B, N, d_model)
        return self.W_o(out)


# Teste rápido
mha = MultiHeadAttention(d_model=64, num_heads=4)
x_test = tf.random.normal((2, 10, 64))
out_test = mha(x_test)
print(f'Entrada: {x_test.shape}  →  Saída MHA: {out_test.shape}')

## 4. Bloco Encoder Completo

1. **MHA** + conexão residual + LayerNorm
2. **FFN** (Dense → ReLU → Dense) + conexão residual + LayerNorm

In [ ]:
class EncoderBlock(keras.layers.Layer):
    def __init__(self, d_model, num_heads, d_ff=None, dropout=0.1, **kwargs):
        super().__init__(**kwargs)
        d_ff = d_ff or 4 * d_model
        self.attn  = MultiHeadAttention(d_model, num_heads, dropout)
        self.ff    = keras.Sequential([
            keras.layers.Dense(d_ff, activation='relu'),
            keras.layers.Dropout(dropout),
            keras.layers.Dense(d_model),
            keras.layers.Dropout(dropout),
        ])
        self.norm1 = keras.layers.LayerNormalization(epsilon=1e-6)
        self.norm2 = keras.layers.LayerNormalization(epsilon=1e-6)
        self.drop  = keras.layers.Dropout(dropout)

    def call(self, x, mask=None, training=False):
        # Sublayer 1: MHA + residual + norm
        x = self.norm1(x + self.drop(self.attn(x, mask, training), training=training))
        # Sublayer 2: FFN + residual + norm
        x = self.norm2(x + self.ff(x, training=training))
        return x


class TransformerEncoder(keras.layers.Layer):
    def __init__(self, vocab_size, d_model, num_heads, num_layers,
                 d_ff=None, max_len=512, dropout=0.1, pad_idx=0, **kwargs):
        super().__init__(**kwargs)
        self.pad_idx   = pad_idx
        self.d_model   = d_model
        self.embedding = keras.layers.Embedding(vocab_size, d_model, mask_zero=(pad_idx==0))
        self.pos_enc   = PositionalEncoding(d_model, max_len)
        self.drop      = keras.layers.Dropout(dropout)
        self.blocks    = [EncoderBlock(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)]

    def call(self, tokens, training=False):
        # Máscara de padding: (B, 1, 1, N)
        pad_mask = tf.cast(tf.equal(tokens, self.pad_idx), tf.float32)[:, tf.newaxis, tf.newaxis, :]
        x = self.drop(self.pos_enc(self.embedding(tokens) * math.sqrt(self.d_model)), training=training)
        for block in self.blocks:
            x = block(x, mask=pad_mask, training=training)
        return x


# Teste
enc = TransformerEncoder(vocab_size=1000, d_model=64, num_heads=4, num_layers=2)
tokens_test = tf.random.uniform((2, 15), minval=1, maxval=1000, dtype=tf.int32)
enc_out = enc(tokens_test)
print(f'Tokens: {tokens_test.shape}  →  Encoder out: {enc_out.shape}')

## 5. Aplicação: NER com WikiANN

Named Entity Recognition — classificação de cada token em PER, ORG, LOC ou O (formato BIO).

```
Tokens → Embedding + PosEnc → [Encoder × N] → Dense → Softmax por token
```

In [ ]:
from datasets import load_dataset

raw = load_dataset('wikiann', 'en')  # parquet nativo, sem script legado
print(raw)
ex = raw['train'][0]
print('\nTokens:', ex['tokens'])
print('NER tags:', ex['ner_tags'])
print('\nRótulos:', raw['train'].features['ner_tags'].feature.names)

In [ ]:
PAD_IDX  = 0
UNK_IDX  = 1
MAX_LEN  = 64

counter = Counter()
for ex in raw['train']:
    counter.update(t.lower() for t in ex['tokens'])

vocab    = ['<PAD>', '<UNK>'] + [w for w, c in counter.most_common() if c >= 2]
word2idx = {w: i for i, w in enumerate(vocab)}
VOCAB_SIZE  = len(vocab)
label_names = raw['train'].features['ner_tags'].feature.names  # O, B-PER, I-PER, B-ORG, I-ORG, B-LOC, I-LOC
NUM_LABELS  = len(label_names)
print(f'Vocabulário: {VOCAB_SIZE}  Rótulos: {NUM_LABELS} → {label_names}')

In [ ]:
def encode_split(split):
    X, Y = [], []
    for ex in raw[split]:
        toks = [word2idx.get(t.lower(), UNK_IDX) for t in ex['tokens']]
        tags = ex['ner_tags']
        toks = toks[:MAX_LEN] + [PAD_IDX] * max(0, MAX_LEN - len(toks))
        tags = tags[:MAX_LEN] + [-1]     * max(0, MAX_LEN - len(tags))
        X.append(toks); Y.append(tags)
    return np.array(X, dtype=np.int32), np.array(Y, dtype=np.int32)

X_train, y_train = encode_split('train')
X_val,   y_val   = encode_split('validation')
X_test,  y_test  = encode_split('test')
print(f'Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}')

In [ ]:
def build_transformer_ner(vocab_size, d_model, num_heads, num_layers, num_labels,
                           max_len=MAX_LEN, dropout=0.1):
    inp = keras.Input(shape=(max_len,), dtype='int32', name='tokens')
    x   = TransformerEncoder(
        vocab_size, d_model, num_heads, num_layers,
        max_len=max_len, dropout=dropout
    )(inp)
    out = keras.layers.Dense(num_labels, activation='softmax', name='ner_head')(x)
    return keras.Model(inp, out)


model = build_transformer_ner(
    vocab_size=VOCAB_SIZE, d_model=128, num_heads=4,
    num_layers=2, num_labels=NUM_LABELS
)
model.summary(line_length=70)

## 6. Treinamento

In [ ]:
# Loss que ignora posições de padding (tag = -1)
class MaskedSparseCategoricalCrossentropy(keras.losses.Loss):
    def __init__(self, pad_label=-1, **kwargs):
        super().__init__(**kwargs)
        self.pad_label = pad_label

    def call(self, y_true, y_pred):
        mask  = tf.cast(tf.not_equal(y_true, self.pad_label), tf.float32)  # (B, N)
        y_true_clipped = tf.maximum(y_true, 0)  # evita índice negativo no one-hot
        loss  = keras.losses.sparse_categorical_crossentropy(y_true_clipped, y_pred)
        return tf.reduce_sum(loss * mask) / (tf.reduce_sum(mask) + 1e-8)


class MaskedAccuracy(keras.metrics.Metric):
    def __init__(self, pad_label=-1, **kwargs):
        super().__init__(**kwargs)
        self.pad_label = pad_label
        self.correct = self.add_weight('correct', initializer='zeros')
        self.total   = self.add_weight('total',   initializer='zeros')

    def update_state(self, y_true, y_pred, sample_weight=None):
        preds = tf.cast(tf.argmax(y_pred, axis=-1), tf.int32)
        mask  = tf.not_equal(y_true, self.pad_label)
        self.correct.assign_add(tf.reduce_sum(tf.cast(tf.equal(preds, y_true) & mask, tf.float32)))
        self.total.assign_add(tf.reduce_sum(tf.cast(mask, tf.float32)))

    def result(self):
        return self.correct / (self.total + 1e-8)

    def reset_state(self):
        self.correct.assign(0.)
        self.total.assign(0.)


BATCH = 32
EPOCHS = 5

steps = len(X_train) // BATCH
schedule = keras.optimizers.schedules.CosineDecay(3e-4, decay_steps=steps * EPOCHS)

model.compile(
    optimizer=keras.optimizers.AdamW(schedule, weight_decay=1e-2),
    loss=MaskedSparseCategoricalCrossentropy(),
    metrics=[MaskedAccuracy(name='acc')]
)

hist = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS, batch_size=BATCH,
    verbose=1
)

In [ ]:
# Avaliação no conjunto de teste
loss_t, acc_t = model.evaluate(X_test, y_test, batch_size=64, verbose=0)
print(f'Test loss: {loss_t:.4f}  Test accuracy: {acc_t:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(hist.history['loss'], marker='o', color='#3b82f6', label='train')
axes[0].plot(hist.history['val_loss'], marker='s', color='#f59e0b', label='val')
axes[0].set(title='Loss', xlabel='Época', ylabel='CrossEntropy'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(hist.history['acc'], marker='o', color='#3b82f6', label='train')
axes[1].plot(hist.history['val_acc'], marker='s', color='#f59e0b', label='val')
axes[1].set(title='Acurácia', xlabel='Época', ylabel='Acc', ylim=(0.5, 1.0)); axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 7. Visualização dos Pesos de Atenção

In [ ]:
def get_encoder_block(model, block_idx=1):
    """Recupera o bloco encoder pelo índice."""
    for layer in model.layers:
        if isinstance(layer, TransformerEncoder):
            return layer.blocks[block_idx]
    return None


def show_attention(sentence: str, layer_idx: int = -1):
    tokens_str = sentence.lower().split()
    token_ids  = [word2idx.get(t, UNK_IDX) for t in tokens_str]
    pad_needed = MAX_LEN - len(token_ids)
    inp = np.array([token_ids + [PAD_IDX] * pad_needed], dtype=np.int32)

    _ = model(inp, training=False)  # forward pass

    block = None
    for layer in model.layers:
        if isinstance(layer, TransformerEncoder):
            block = layer.blocks[layer_idx]

    if block is None:
        print('Bloco não encontrado'); return

    attn = block.attn.last_attn.numpy()  # (1, h, N, N)
    N = len(tokens_str)
    attn = attn[0, :, :N, :N]           # (h, N, N)
    num_heads = attn.shape[0]

    fig, axes = plt.subplots(1, num_heads, figsize=(4 * num_heads, 4))
    if num_heads == 1: axes = [axes]
    for h_idx, ax in enumerate(axes):
        sns.heatmap(attn[h_idx], annot=True, fmt='.2f', cmap='Blues',
                    xticklabels=tokens_str, yticklabels=tokens_str,
                    ax=ax, cbar=False, square=True, annot_kws={'size': 8})
        ax.set_title(f'Cabeça {h_idx+1}', fontsize=10)
        ax.tick_params(axis='both', labelsize=8)
    plt.suptitle(f'Pesos de atenção — "{sentence}"', fontsize=11, y=1.02)
    plt.tight_layout(); plt.show()


show_attention('EU lives in New York and works for Google')

## 8. Comparação: Transformer vs. BiLSTM

In [ ]:
def build_bilstm_ner(vocab_size, embed_dim, hidden_dim, num_layers, num_labels, max_len=MAX_LEN, dropout=0.1):
    inp = keras.Input(shape=(max_len,), dtype='int32')
    x   = keras.layers.Embedding(vocab_size, embed_dim, mask_zero=True)(inp)
    x   = keras.layers.Dropout(dropout)(x)
    for i in range(num_layers):
        ret_seq = True  # sempre retornar sequência para classificação por token
        x = keras.layers.Bidirectional(
            keras.layers.LSTM(hidden_dim, return_sequences=ret_seq,
                              dropout=dropout if i < num_layers - 1 else 0)
        )(x)
    out = keras.layers.Dense(num_labels, activation='softmax')(x)
    return keras.Model(inp, out)


lstm_model = build_bilstm_ner(
    vocab_size=VOCAB_SIZE, embed_dim=128, hidden_dim=64,
    num_layers=2, num_labels=NUM_LABELS
)

print(f"Parâmetros Transformer: {model.count_params():,}")
print(f"Parâmetros BiLSTM:      {lstm_model.count_params():,}")

In [ ]:
lstm_model.compile(
    optimizer=keras.optimizers.AdamW(3e-4, weight_decay=1e-2),
    loss=MaskedSparseCategoricalCrossentropy(),
    metrics=[MaskedAccuracy(name='acc')]
)

hist_lstm = lstm_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS, batch_size=BATCH, verbose=1
)

_, lstm_acc = lstm_model.evaluate(X_test, y_test, batch_size=64, verbose=0)
print(f'\nBiLSTM  Test accuracy: {lstm_acc:.4f}')
print(f'Transformer Test accuracy: {acc_t:.4f}')

In [ ]:
epochs = range(1, EPOCHS + 1)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(epochs, hist.history['loss'],      marker='o', color='#3b82f6', label='Transformer')
axes[0].plot(epochs, hist_lstm.history['loss'], marker='s', color='#ef4444', label='BiLSTM')
axes[0].set(title='Loss de treinamento', xlabel='Época', ylabel='CrossEntropy')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs, hist.history['val_acc'],      marker='o', color='#3b82f6', label='Transformer val')
axes[1].plot(epochs, hist_lstm.history['val_acc'], marker='s', color='#ef4444', label='BiLSTM val')
axes[1].axhline(acc_t,    color='#3b82f6', linestyle='--', alpha=0.6, label=f'Transformer test ({acc_t:.3f})')
axes[1].axhline(lstm_acc, color='#ef4444', linestyle='--', alpha=0.6, label=f'BiLSTM test ({lstm_acc:.3f})')
axes[1].set(title='Acurácia de validação', xlabel='Época', ylabel='Acc', ylim=(0.5, 1.0))
axes[1].legend(fontsize=8); axes[1].grid(True, alpha=0.3)

plt.suptitle('Transformer vs. BiLSTM — NER (CoNLL-2003)', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

## 9. Resumo

| Componente | Keras | Equivalente PyTorch |
|---|---|---|
| Embedding | `keras.layers.Embedding` | `nn.Embedding` |
| Multi-Head Attention | `keras.layers.MultiHeadAttention` (ou custom) | `nn.MultiheadAttention` |
| Layer Norm | `keras.layers.LayerNormalization` | `nn.LayerNorm` |
| FFN | `keras.Sequential` | `nn.Sequential` |
| Modelo completo | `keras.Model` (functional API) | `nn.Module` |

**Vantagens do Transformer sobre BiLSTM:**
- Contexto global em uma única operação (sem propagação sequencial)
- Treinamento totalmente paralelizável
- Sem vanishing gradient no caminho da atenção

**Próxima aula:** BERT — pré-treino com Masked Language Modeling e fine-tuning.